# Limpieza de Datos para XRP 🧹

### Paso 1: Carga y Exploración Inicial

¿Por qué lo hacemos? Antes de limpiar, necesitamos conocer la estructura, los tipos de datos y si faltan valores.

In [2]:
import pandas as pd

# Cargar el archivo
df = pd.read_csv('items/XRP_Data.csv')

# Visualizar las primeras filas y la info técnica
print("Primeras filas:")

display(df.head())
print("\nInformación del DataFrame:")
df.info()

Primeras filas:


,Date,Ticker,Currency,Open,High,Low,Close,Volume
0,2017-11-09,XRP-USD,USD,0.217911,0.221791,0.214866,0.217488,147916992
1,2017-11-10,XRP-USD,USD,0.218256,0.219068,0.205260,0.206483,141032992
2,2017-11-11,XRP-USD,USD,0.205948,0.214456,0.205459,0.210430,134503008
3,2017-11-12,XRP-USD,USD,0.210214,0.210214,0.195389,0.197339,251175008
4,2017-11-13,XRP-USD,USD,0.197472,0.204081,0.197456,0.203442,132567000



Información del DataFrame:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3044 entries, 0 to 3043
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Date      3044 non-null   object 
 1   Ticker    3044 non-null   object 
 2   Currency  3044 non-null   object 
 3   Open      3044 non-null   float64
 4   High      3044 non-null   float64
 5   Low       3044 non-null   float64
 6   Close     3044 non-null   float64
 7   Volume    3044 non-null   int64  
dtypes: float64(4), int64(1), object(3)
memory usage: 190.4+ KB


### Paso 2: Conversión del Formato de Fecha

¿Por qué lo hacemos? Por defecto, las fechas se cargan como "objetos" (texto). Para hacer análisis de series temporales, cálculos de rendimiento o gráficos por años/meses, necesitamos que Python las reconozca como objetos datetime.

In [3]:
# Convertir la columna Date a formato datetime
df['Date'] = pd.to_datetime(df['Date'])

# Verificar el cambio
print(f"Nuevo tipo de dato de 'Date': {df['Date'].dtype}")

Nuevo tipo de dato de 'Date': datetime64[ns]


### Paso 3: Manejo de Valores Nulos (Missing Values)

¿Por qué lo hacemos? Yahoo Finance a veces tiene "huecos" por días en los que no hubo registro o falló la API. Los valores nulos rompen los algoritmos matemáticos.

In [4]:
# Contar valores nulos por columna
print("Valores nulos encontrados:")
print(df.isnull().sum())

# Si encontraras nulos, la mejor práctica en finanzas es el 'Forward Fill' 
# (llenar con el precio del día anterior) en lugar de borrar la fila.
df = df.fillna(method='ffill')

Valores nulos encontrados:
Date        0
Ticker      0
Currency    0
Open        0
High        0
Low         0
Close       0
Volume      0
dtype: int64


C:\Users\delhy.py\AppData\Local\Temp\ipykernel_6828\422995742.py:7: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df = df.fillna(method='ffill')


###  Paso 4: Detección y Eliminación de Duplicados
¿Por qué lo hacemos? Los registros duplicados sesgan los promedios y el volumen total de transacciones, inflando artificialmente las métricas.

In [5]:
# Buscar filas idénticas
duplicados = df.duplicated().sum()
print(f"Registros duplicados: {duplicados}")

# Eliminar si existen (manteniendo la primera aparición)
if duplicados > 0:
    df = df.drop_duplicates()

Registros duplicados: 0


### Paso 5: Ordenamiento Cronológico e Indexación

¿Por qué lo hacemos? En el trading, el orden de los factores sí altera el producto. Para calcular variaciones diarias (porcentajes de cambio), los datos deben estar ordenados del más antiguo al más reciente. Establecer la fecha como índice facilita las búsquedas.

In [6]:
# Ordenar por fecha por si acaso vienen desordenados
df = df.sort_values(by='Date')

# Establecer la fecha como índice
df.set_index('Date', inplace=True)

print("Datos listos y ordenados:")
display(df.head())

Datos listos y ordenados:


,Ticker,Currency,Open,High,Low,Close,Volume
Date,,,,,,,
2017-11-09,XRP-USD,USD,0.217911,0.221791,0.214866,0.217488,147916992
2017-11-10,XRP-USD,USD,0.218256,0.219068,0.205260,0.206483,141032992
2017-11-11,XRP-USD,USD,0.205948,0.214456,0.205459,0.210430,134503008
2017-11-12,XRP-USD,USD,0.210214,0.210214,0.195389,0.197339,251175008
2017-11-13,XRP-USD,USD,0.197472,0.204081,0.197456,0.203442,132567000


### Paso 6: Verificación de Consistencia (Outliers)

¿Por qué lo hacemos? A veces hay errores en los decimales (ej. un precio que dice $100 cuando XRP vale $1). Un chequeo rápido de los valores máximos y mínimos nos ayuda a detectar errores de carga.

In [7]:
# Resumen estadístico
# Buscamos valores que no tengan sentido (ej. un Low de 0 o un High negativo)
display(df.describe())

,Open,High,Low,Close,Volume
count,3044.000000,3044.000000,3044.000000,3044.000000,3.044000e+03
mean,0.810512,0.838788,0.780777,0.810917,2.883614e+09
std,0.745611,0.772514,0.718544,0.745628,3.871341e+09
min,0.140524,0.146911,0.115093,0.139635,1.002940e+08
25%,0.334478,0.345358,0.324965,0.334992,9.533236e+08
50%,0.516728,0.527266,0.501907,0.517056,1.625303e+09
75%,0.859550,0.893341,0.823169,0.860899,3.192528e+09
max,3.555637,3.841940,3.430836,3.555765,5.172338e+10


### Paso 7 Auditoría de Integridad Lógica y Valores No Negativos

¿Por qué lo hacemos? En el análisis financiero, existen reglas físicas inquebrantables. Por ejemplo, el precio más bajo de un día (Low) nunca puede ser superior al más alto (High). Asimismo, en un mercado real, los precios y el volumen de transacciones no pueden ser negativos ni cero; de ser así, estaríamos ante un error de "ruido" en la captura de datos que arruinaría cualquier modelo predictivo.

**Validaciones incluidas:**

* Jerarquía de Precios: Verificación de que $Low \leq \{Open, Close\} \leq High$.
* Positividad: Asegurar que $Precio > 0$ y $Volumen \geq 0$.
* Integridad de Tipos: Confirmar que no hay valores no numéricos en las columnas críticas.

In [8]:
# 1. Definir columnas de interés
cols_precio = ['Open', 'High', 'Low', 'Close']

# 2. Verificar Precios Negativos o Cero
# Buscamos si hay algún valor menor o igual a 0 en las columnas de precio
precios_invalidos = df[(df[cols_precio] <= 0).any(axis=1)]

print(f"Registros con precios inválidos: {len(precios_invalidos)}")
if len(precios_invalidos) > 0:
    print("Revisar las siguientes fechas:", precios_invalidos.index.tolist())

# 3. Verificar Volumen Negativo
volumen_invalido = df[df['Volume'] < 0]

print(f"Registros con volumen negativo: {len(volumen_invalido)}")

# 4. Verificar Jerarquía Lógica (High y Low)
# El High debe ser el precio más alto del día
error_high = df[(df['High'] < df['Open']) | (df['High'] < df['Close']) | (df['High'] < df['Low'])]

# El Low debe ser el precio más bajo del día
error_low = df[(df['Low'] > df['Open']) | (df['Low'] > df['Close']) | (df['Low'] > df['High'])]

print(f"Inconsistencias en el precio Máximo (High): {len(error_high)}")
print(f"Inconsistencias en el precio Mínimo (Low): {len(error_low)}")

# 5. Conclusión final de la auditoría
if len(precios_invalidos) == 0 and len(volumen_invalido) == 0 and len(error_high) == 0 and len(error_low) == 0:
    print("\n✅ ÉXITO: Los datos han pasado todas las pruebas de seguridad lógica.")
else:
    print("\n❌ ADVERTENCIA: Se detectaron anomalías en el dataset.")

Registros con precios inválidos: 0
Registros con volumen negativo: 0
Inconsistencias en el precio Máximo (High): 0
Inconsistencias en el precio Mínimo (Low): 0

✅ ÉXITO: Los datos han pasado todas las pruebas de seguridad lógica.
